# Medical Chatbot — Research Notebook (modernized)
End-to-end walkthrough of the RAG pipeline: gather up-to-date data → chunk → embed (BGE) → build Pinecone index → retrieve → answer with a safety-aware prompt. Run this from the `research/` folder; the first cell wires up imports from the project root.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # make `src`, store_index, app importable
os.chdir(os.path.abspath(".."))          # run from project root so DATA_DIR='data/' works
from dotenv import load_dotenv
load_dotenv()
from src.config import Config
config = Config.from_env()
config.require_keys()
os.environ["PINECONE_API_KEY"] = config.pinecone_api_key
os.environ["OPENAI_API_KEY"] = config.openai_api_key
print("Index:", config.index_name, "| Model:", config.llm_model, "| Embeddings:", config.embedding_model)

## 1. Gather up-to-date data from all enabled sources
Curated (built-in, cited) + drop-in files in `data/` + live WHO/CDC/NIH/MedlinePlus.

In [ ]:
from src import data_sources, helper
from store_index import summarize
docs = data_sources.gather_documents(config)
print("Documents by source:", summarize(docs))
print("Total documents:", len(docs))
docs[0].metadata

## 2. Reduce metadata and split into chunks

In [ ]:
minimal_docs = helper.filter_to_minimal_docs(docs)
chunks = helper.text_split(minimal_docs, config)
print("Number of chunks:", len(chunks))
chunks[0]

## 3. Embeddings (BAAI/bge-small-en-v1.5, 384-dim)
First run downloads the model (~130 MB).

In [ ]:
embedding = helper.download_embeddings(config)
vector = embedding.embed_query("What are the warning signs of a stroke?")
print("Vector length:", len(vector))

## 4. (Re)create the Pinecone index and upsert
This deletes and rebuilds the index so it only contains current BGE vectors.

In [ ]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from store_index import recreate_index
pc = Pinecone(api_key=config.pinecone_api_key)
recreate_index(pc, config)
docsearch = PineconeVectorStore.from_documents(
    documents=chunks, embedding=embedding, index_name=config.index_name
)
print("Index built:", config.index_name)

## 5. Retrieve relevant chunks

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": config.retriever_k})
retriever.invoke("What are the warning signs of a stroke?")

## 6. Answer questions with the safety-aware RAG chain

In [ ]:
from app import build_rag_chain
rag_chain = build_rag_chain(config)
for q in ["What are the warning signs of a stroke?",
          "What is a normal resting heart rate?",
          "How can I prevent type 2 diabetes?"]:
    print("Q:", q)
    print(rag_chain.invoke({"input": q})["answer"])
    print("-" * 80)